# ML 2 C 26 — Colab runbook

Single-upload Plan 2. This notebook is a wrapper around the tested Python modules, not an executed competition experiment. Upload and extract the delivered ZIP to `/content/ML-2-C-26`, or point PROJECT at its extracted folder. Read `WORKFLOW.md` first.

In [ ]:
from pathlib import Path
import subprocess, sys, os
PROJECT = Path("/content/ML-2-C-26")
assert (PROJECT / "requirements.txt").is_file(), "Extract the project ZIP and set PROJECT correctly."
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
def run(*args):
    subprocess.run([sys.executable, *args], cwd=PROJECT, check=True)
print(PROJECT)

In [ ]:
run("-m", "pip", "install", "-r", "requirements.txt")
run("-m", "pytest", "-q")

In [ ]:
run("-m", "scripts.run_smoke", "--out", "runs/colab_smoke")

## Real data — disabled by default
Mount your own Drive only when ready. Use the already obtained official dataset, not external business data. Correct the paths below. A real-data probe estimates cost; it does not train the final model.

In [ ]:
RUN_REAL_DATA = False
if RUN_REAL_DATA:
    from google.colab import drive
    drive.mount("/content/drive")
TRAIN = Path("/content/drive/MyDrive/student_resource/dataset/train")
TEST = Path("/content/drive/MyDrive/student_resource/dataset/test")

In [ ]:
if RUN_REAL_DATA:
    for f in ["train_source1.tsv", "train_source2.tsv", "train_source3.tsv", "train_ground_truth.tsv"]:
        assert (TRAIN / f).is_file(), f"Fix dataset path: {TRAIN / f}"
    run("-m", "ml2", "probe-index", "--targets", str(TRAIN/"train_source2.tsv"), str(TRAIN/"train_source3.tsv"), "--limit-per-file", "10000", "--out", "runs/probe10k.sqlite")

## Full run gate
Review the probe JSON. Repeat at 100,000 rows and estimate full disk/time/RAM before enabling this. Probe indexes cannot be used for final training. Large indexes belong on local scratch SSD rather than a slow mounted Drive filesystem; save final artifacts/checkpoints separately.

In [ ]:
RUN_FULL = False
if RUN_REAL_DATA and RUN_FULL:
    run("-m", "ml2", "index", "--targets", str(TRAIN/"train_source2.tsv"), str(TRAIN/"train_source3.tsv"), "--out", "runs/train_targets.sqlite")
    run("-m", "ml2", "train", "--index", "runs/train_targets.sqlite", "--queries", str(TRAIN/"train_source1.tsv"), "--truth", str(TRAIN/"train_ground_truth.tsv"), "--out", "runs/baseline_v1")

Inspect the real-data report, split distribution and artifacts before test inference. CARE is experimental; pair-only is the default. Follow the test-index, prediction and validation commands in README. No final submission ZIP is produced by this runbook.